# Импорты библиотек

In [10]:
import plotly.express as px
import numpy
from scipy.integrate import quad

# Входные данные

$\displaystyle
f(x) = \frac{1}{\sqrt{1 - 0.25 \sin^2 x}} \\
f''(x) = \frac{16\cos 2x - 4\cos 2x \sin^2 x + 3\sin^2 2x}{2 (4 - \sin^2 x)^{\textstyle \frac{5}{2}}} \\
\int_{0}^{\textstyle \frac{\pi}{3}} \frac{dx}{\sqrt{1 - 0.25 \sin^2 x}} \\
$

In [11]:
def f(x: float) -> float:
    return 1 / numpy.sqrt(1 - 0.25 * numpy.sin(x) ** 2)

def f_2(x: float) -> float:
    return (16 * numpy.cos(2 * x) - 4 * numpy.cos(2 * x) * numpy.sin(x) ** 2 + 3 * numpy.sin(2 * x) ** 2) / (2 * (4 - numpy.sin(x) ** 2) ** 2.5)

a = 0
b = numpy.pi / 3

# Дискретный сплайн

## Определение

$\displaystyle
S_D^i(x) = \sum_{j = 0}^3 a_{ij} (x - x_i)^j, ~ \forall x \in [x_i; x_{i + 1}] \\
S_D(x) = \begin{cases}
    S_D^0(x), & x \in [x_0; x_1] \\
    S_D^1(x), & x \in [x_1; x_2] \\
    \ldots \\
    S_D^{n - 1}(x), & x \in [x_{n - 1}; x_n] \\
\end{cases}
$ -- дискретный сплайн, если: <br>
1) $\displaystyle
   \forall \varepsilon_i > 0, i = \overline{1, n - 1} \Rightarrow \frac{S_D^i(x_i + \varepsilon_i) - S_D^i(x_i - \varepsilon_i)}{2\varepsilon_i} = \frac{S_D^{i - 1}(x_i + \varepsilon_i) - S_D^{i - 1}(x_i - \varepsilon_i)}{2\varepsilon_i}
   $
2) $\displaystyle
   \forall \varepsilon_i > 0, i = \overline{1, n - 1} \Rightarrow \frac{S_D^i(x_i + \varepsilon_i) - 2S_D^i(x_i) + S_D^i(x_i - \varepsilon_i)}{\varepsilon^2_i} = \frac{S_D^{i - 1}(x_i + \varepsilon_i) - 2S_D^{i - 1}(x_i) + S_D^{i - 1}(x_i - \varepsilon_i)}{\varepsilon^2_i}
   $
3) $S_D(x_i) = f_i, ~ \forall i = \overline{0, n}$ 

$\displaystyle
M_i = \frac{S_D^i(x_i + \varepsilon_i) - 2S_D^i(x_i) + S_D^i(x_i - \varepsilon_i)}{\varepsilon_i^2} \\
S_D^i(x) = f_i (1 - t) + f_{i + 1} t - \frac{t(1 - t)}{6} h_i^2 ((2 - t) M_i + (1 + t) M_{i + 1}), ~~ h_i = x_{i + 1} - x_i, ~ t = \frac{x - x_i}{h_i}
$

In [12]:
def S_D_i(x: float, i: int, x_array: numpy.ndarray, f: numpy.ndarray, M: numpy.ndarray) -> float:
    h = x_array[i + 1] - x_array[i]
    t = (x - x_array[i]) / h

    return f[i] * (1 - t) + f[i + 1] * t - t * (1 - t) / 6 * h ** 2 * ((2 - t) * M[i] + (1 + t) * M[i + 1])

## Метод монотонной прогонки

In [13]:
def monotonous_reduction(A: numpy.array, B: numpy.array, C: numpy.array, F: numpy.array) -> numpy.array:
    n = len(A) - 1

    alpha = numpy.zeros(n + 1)
    beta = numpy.zeros(n + 1)
    U = numpy.zeros(n + 1)

    alpha[1] = -C[0] / B[0]
    beta[1] = F[0] / B[0]

    for i in range(1, n):
        alpha[i + 1] = -C[i] / (B[i] + alpha[i] * A[i])
        beta[i + 1] = (F[i] - beta[i] * A[i]) / (B[i] + alpha[i] * A[i])

    U[n] = (F[n] - beta[n] * A[n]) / (B[n] + alpha[n] * A[n])

    for i in range(n - 1, -1, -1):
        U[i] = alpha[i + 1] * U[i + 1] + beta[i + 1]

    return U

## Система

$\displaystyle
\mu_i = \frac{h_{i - 1}}{h_i + h_{i - 1}}, ~ \lambda_i = \frac{h_i}{h_i + h_{i - 1}} \\
\mu_i \left( 1 - \frac{\varepsilon_i^2}{h_{i - 1}^2} \right) M_{i - 1} + \left( 2 + \frac{\varepsilon_i^2}{h_i h_{i - 1}} \right) M_i + \lambda_i \left( 1 - \frac{\varepsilon_i^2}{h_i^2} \right) M_{i + 1} = \frac{6}{h_i + h_{i - 1}} \left( \frac{f_{i + 1} - f_i}{h_i} - \frac{f_i - f_{i - 1}}{h_{i - 1}} \right), ~~~ i = \overline{1, n - 1}
$

При II-х граничных условиях: <br>
$\displaystyle
S_D''(a) = f_0'', ~~ S_D''(b) = f_n'' \Rightarrow M_0 = f_0'', ~~ M_n = f_n''
$

Тогда система имеет вид: <br>
$\displaystyle
\left( \begin{array}{ccccccccc}
    1 & 0 & 0 & 0 & \cdots & 0 & 0 & 0 & 0 \\
    \displaystyle \mu_1 \left( 1 - \frac{\varepsilon_1^2}{h_0^2} \right) & \displaystyle 2 + \frac{\varepsilon_1^2}{h_1 h_0} & \displaystyle \lambda_1 \left( 1 - \frac{\varepsilon_1^2}{h_1^2} \right) & 0 & \cdots & 0 & 0 & 0 & 0 \\
    0 & \displaystyle \mu_2 \left( 1 - \frac{\varepsilon_2^2}{h_1^2} \right) & \displaystyle 2 + \frac{\varepsilon_2^2}{h_2 h_1} & \displaystyle \lambda_2 \left( 1 - \frac{\varepsilon_2^2}{h_2^2} \right) & \cdots & 0 & 0 & 0 & 0 \\
    \vdots & \vdots & \vdots & \vdots & \ddots & \vdots & \vdots & \vdots & \vdots \\
    0 & 0 & 0 & 0 & \cdots & \displaystyle \mu_{n - 2} \left( 1 - \frac{\varepsilon_{n - 2}^2}{h_{n - 3}^2} \right) & \displaystyle 2 + \frac{\varepsilon_{n - 2}^2}{h_{n - 2} h_{n - 3}} & \displaystyle \lambda_{n - 2} \left( 1 - \frac{\varepsilon_{n - 2}^2}{h_{n - 2}^2} \right) & 0 \\
    0 & 0 & 0 & 0 & \cdots & 0 & \displaystyle \mu_{n - 1} \left( 1 - \frac{\varepsilon_{n - 1}^2}{h_{n - 2}^2} \right) & \displaystyle 2 + \frac{\varepsilon_{n - 1}^2}{h_{n - 1} h_{n - 2}} & \displaystyle \lambda_{n - 1} \left( 1 - \frac{\varepsilon_{n - 1}^2}{h_{n - 1}^2} \right) \\
    0 & 0 & 0 & 0 & \cdots & 0 & 0 & 0 & 1 \\
\end{array} \right) \cdot \left( \begin{array}{c}
    M_0 \\ M_1 \\ M_2 \\ \vdots \\ M_{n - 2} \\ M_{n - 1} \\ M_{n}
\end{array} \right) = \left( \begin{array}{c}
    f_0'' \\
    \displaystyle \frac{6}{h_1 + h_0} \left( \frac{f_2 - f_1}{h_1} - \frac{f_1 - f_0}{h_0} \right) \\
    \displaystyle \frac{6}{h_2 + h_1} \left( \frac{f_3 - f_2}{h_2} - \frac{f_2 - f_1}{h_1} \right) \\
    \vdots \\
    \displaystyle \frac{6}{h_{n - 2} + h_{n - 3}} \left( \frac{f_{n - 1} - f_{n - 2}}{h_{n - 2}} - \frac{f_{n - 2} - f_{n - 3}}{h_{n - 3}} \right) \\
    \displaystyle \frac{6}{h_{n - 1} + h_{n - 2}} \left( \frac{f_n - f_{n - 1}}{h_{n - 1}} - \frac{f_{n - 1} - f_{n - 2}}{h_{n - 2}} \right) \\
    f_n''
\end{array} \right)
$

In [14]:
def solve_with_2_conditions(x: numpy.ndarray, f: numpy.ndarray, epsilon: numpy.ndarray, n: int, f_0: float, f_n: float) -> numpy.ndarray:
    A = numpy.zeros(n + 1)
    B = numpy.zeros(n + 1)
    C = numpy.zeros(n + 1)
    F = numpy.zeros(n + 1)

    B[0] = 1
    F[0] = f_0

    B[n] = 1
    F[n] = f_n

    for i in range(1, n):
        h_i_1 = x[i] - x[i - 1]
        h_i = x[i + 1] - x[i]

        mu_i = h_i_1 / (h_i + h_i_1)
        lambda_i = h_i / (h_i + h_i_1)

        A[i] = mu_i * (1 - epsilon[i] ** 2 / h_i_1 ** 2)
        B[i] = 2 + epsilon[i] ** 2 / (h_i * h_i_1)
        C[i] = lambda_i * (1 - epsilon[i] ** 2 / h_i ** 2)
        F[i] = 6 / (h_i + h_i_1) * ((f[i + 1] - f[i]) / h_i - (f[i] - f[i - 1]) / h_i_1)

    M = monotonous_reduction(A, B, C, F)
    return M

## Определение сплайна

In [15]:
n = 10
x = numpy.linspace(a, b, n + 1)
epsilon = numpy.zeros(n + 1)
epsilon[1:n] = x[1] - x[0]

f_0 = f_2(a)
f_n = f_2(b)

M = solve_with_2_conditions(x, f(x), epsilon, n, f_0, f_n)

def S_D(x: float, x_array: numpy.ndarray) -> float:
    i = numpy.searchsorted(x_array, x) - 1
    return S_D_i(x, i, x_array, f(x_array), M)

## Визуализация

In [16]:
xs = numpy.linspace(a, b, 1000)
ys = f(xs)
S_Ds = S_D(xs, x)
R = abs(f(xs) - S_D(xs, x))

fig = px.line(x=xs, y=ys)
fig.update_traces(name="f(x)", showlegend=True)
fig.add_scatter(x=xs, y=S_Ds, name="S_D(x)")
fig.show()

fig = px.line(x=xs, y=R)
fig.update_traces(name="R(x)")
fig.update_yaxes(exponentformat="power")
fig.show()

# Интегрирование

$\displaystyle
\int_{0}^{\textstyle \frac{\pi}{3}} S_D(x) dx = \sum_{i = 0}^{n - 1} \int_{x_i}^{x_{i + 1}} S_D^i(x) dx = \left| t = \frac{x - x_i}{h_i} ~~ x = x_i + h_i t ~~ dx = h_i dt \right| = \sum_{i = 0}^{n - 1} h_i \int_{0}^{1} \left( f_i (1 - t) + f_{i + 1} t - \frac{t(1 - t)}{6} h_i^2 ((2 - t) M_i + (1 + t) M_{i + 1}) \right) dt = \sum_{i = 0}^{n - 1} h_i \left. \left( f_i t - \frac{f_i}{2} t^2 + \frac{f_{i + 1}}{2} t^2 - \frac{h_i^2}{6} \left( M_i \left( \frac{t^4}{4} - t^3 + t^2 \right) + M_{i + 1} \left( \frac{t^2}{2} - \frac{t^4}{4} \right) \right) \right) \right|_0^1 = \sum_{i = 0}^{n - 1} h_i \left( f_i - \frac{f_i}{2} + \frac{f_{i + 1}}{2} + \frac{h_i^2}{6} \left( \frac{M_i}{4} + \frac{M_{i + 1}}{4} \right) \right) = \sum_{i = 0}^{n - 1} \left( \frac{h_i}{2} (f_i + f_{i + 1}) - \frac{h_i^3}{24} (M_i + M_{i + 1}) \right)
$

In [17]:
def int_S_D(x: numpy.ndarray, f: numpy.ndarray, M: numpy.ndarray) -> float:
    return numpy.sum([
        (x[i + 1] - x[i]) / 2 * (f[i] + f[i + 1]) - (x[i + 1] - x[i]) ** 3 / 24 * (M[i] + M[i + 1])
        for i in range(n)
    ])

## Результат

In [18]:
int_f, _ = quad(f, a, b, epsrel=1e-10, epsabs=1e-10)
int_S_D = int_S_D(x, f(x), M)
print(int_f, int_S_D, abs(int_f - int_S_D))

1.0895506700518853 1.0895520250726016 1.3550207162982275e-06
